<a href="https://colab.research.google.com/github/AnHgPham/AnHgPham/blob/main/Copy_Folder_Google_Drive_to_Google_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder Google Drive to Google Drive

In [6]:
#@title Nhập Liệu
from ipywidgets import widgets

dest_text = widgets.Text(
    description="Drive của bạn",
    placeholder='Nhập đường link thư mục Google Drive **đích** của bạn'
)
source_text = widgets.Text(
    description="Drive nguồn",
    placeholder='Nhập đường link thư mục Google Drive **nguồn**'
)
from_page_text = widgets.IntText(
    description="Từ trang",
    value=0
)
to_page_text = widgets.IntText(
    description="Đến trang",
    value=0
)
max_download_size_text = widgets.FloatText(
    description="Dung lượng tối đa (GB)",
    value=700
)
exclude_str_text = widgets.Text(
    description="Loại trừ (tệp/thư mục chứa)",
    value=""
)

display(dest_text)
display(source_text)
display(from_page_text)
display(to_page_text)
display(max_download_size_text)
display(exclude_str_text)


Text(value='', description='Drive của bạn', placeholder='Nhập đường link thư mục Google Drive **đích** của bạn…

Text(value='', description='Drive nguồn', placeholder='Nhập đường link thư mục Google Drive **nguồn**')

IntText(value=0, description='Từ trang')

IntText(value=0, description='Đến trang')

FloatText(value=700.0, description='Dung lượng tối đa (GB)')

Text(value='', description='Loại trừ (tệp/thư mục chứa)')

In [7]:
#@title Chạy
import os
import time
import re
import sys
import io
import random  # Thêm import random
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload  # Sửa lại MediaIoBaseUpload
from googleapiclient.errors import HttpError  # Thêm import HttpError
from google.colab import auth

class DownloadFromDrive:
    def __init__(self):
        self._total_size = 0
        self._limit_size = 0
        self.excluded_strings = []

    def get_user_credential(self):
        auth.authenticate_user()
        drive_service = build('drive', 'v3')
        return drive_service

    def get_childs_from_folder(self, drive_service, folder_id, from_page, to_page):
        files = []
        page_token = None
        query = f"'{folder_id}' in parents and trashed = false"
        if self.excluded_strings:
            not_contains_query = " and ".join([f"not name contains '{ext}'" for ext in self.excluded_strings])
            query += f" and ({not_contains_query})"

        pages = 0
        while True:
            try:
                pages += 1
                response = drive_service.files().list(
                    q=query,
                    orderBy='name, createdTime',
                    fields='files(id, name, mimeType, size), nextPageToken',
                    pageToken=page_token,
                    supportsAllDrives=True,
                    includeItemsFromAllDrives=True
                ).execute()

                if (from_page < pages <= to_page) or to_page == 0:
                    files.extend(response.get('files', []))

                page_token = response.get('nextPageToken', None)
                if page_token is None or (pages >= to_page > 0):
                    break
            except Exception as e:
                print(f"Đã xảy ra lỗi: {str(e)}")
                break

        print(f"📂 Tổng số tệp: {len(files)}")
        return files

    def copy_file(self, drive_service, dest_folder_id, source_file):
        if source_file['mimeType'] != 'application/vnd.google-apps.folder':
            if not self.check_if_exists(drive_service, dest_folder_id, source_file['name']):
                file_id = source_file['id']
                file_name = source_file['name']
                mime_type = source_file['mimeType']
                file_size = int(source_file.get('size', 0))
                size_mb = file_size / (1024 * 1024)
                self._total_size += size_mb

                max_retries = 5  # Số lần thử tối đa
                retry_count = 0

                while retry_count < max_retries:
                    try:
                        if file_name.endswith('.zip'):
                            # Tải xuống tệp ZIP
                            request = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
                            fh = io.BytesIO()
                            downloader = MediaIoBaseDownload(fh, request)
                            done = False
                            while not done:
                                status, done = downloader.next_chunk()
                                print(f"⬇️ Đang tải xuống {int(status.progress() * 100)}% tệp {file_name}")

                            fh.seek(0)

                            # Giải nén tệp ZIP nếu cần mật khẩu
                            # Hiện tại, đoạn mã chỉ sao chép tệp ZIP trực tiếp với mật khẩu "1"

                            # Tải lên tệp ZIP đến thư mục đích
                            media = MediaIoBaseUpload(fh, mimetype='application/zip', resumable=True)
                            file_metadata = {'name': file_name, 'parents': [dest_folder_id]}
                            drive_service.files().create(
                                body=file_metadata,
                                media_body=media,
                                supportsAllDrives=True
                            ).execute()

                            print(f"✅ [{file_name}] đã sao chép (ZIP). Kích thước {size_mb:.2f} MB.")
                        else:
                            # Sao chép tệp thông thường
                            drive_service.files().copy(
                                body={'parents': [dest_folder_id]},
                                fileId=file_id,
                                supportsAllDrives=True
                            ).execute()
                            print(f"✅ [{file_name}] đã sao chép. Kích thước {size_mb:.2f} MB.")

                        # Kiểm tra dung lượng giới hạn
                        if self._total_size >= (self._limit_size * 1024):
                            self.on_total_size_exceeded(f"⚠️ Tổng dung lượng vượt quá {self._limit_size} GB. Dừng chương trình.")
                        break  # Thoát khỏi vòng lặp nếu thành công
                    except HttpError as error:
                        error_reason = error.error_details[0]['reason'] if error.error_details else ''
                        if error.resp.status in [403, 429]:
                            if 'userRateLimitExceeded' in error_reason or 'downloadQuotaExceeded' in error_reason:
                                # Tính toán thời gian chờ tăng dần
                                sleep_time = (2 ** retry_count) + random.uniform(0, 1)
                                print(f"⚠️ Gặp lỗi giới hạn tần suất. Đợi {sleep_time:.2f} giây trước khi thử lại lần {retry_count + 1}...")
                                time.sleep(sleep_time)
                                retry_count += 1
                            else:
                                print(f"❌ Lỗi không thể phục hồi: {error}")
                                break
                        else:
                            print(f"❌ Lỗi khác: {error}")
                            break
                else:
                    print(f"❌ Không thể sao chép [{file_name}] sau {max_retries} lần thử.")
            else:
                print(f"⚠️ [{source_file['name']}] đã tồn tại trong thư mục đích.")
        else:
            # Xử lý thư mục con
            source_files = self.get_childs_from_folder(drive_service, source_file['id'], 0, 0)
            if source_files:
                print(f"📂 Bắt đầu sao chép thư mục {source_file['name']}")
                sub_folder_id = self.create_folder(drive_service, dest_folder_id, source_file['name'])
                self.copy_multiple_files(drive_service, sub_folder_id, source_files)
                print(f"✅ Hoàn thành sao chép thư mục {source_file['name']}")

    def create_folder(self, drive_service, dest_folder_id, sub_folder_name):
        sub_folder_inf = {
            'name': sub_folder_name,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [dest_folder_id]
        }

        exist_folder_id = self.check_if_exists(drive_service, dest_folder_id, sub_folder_name)
        if not exist_folder_id:
            try:
                folder = drive_service.files().create(
                    body=sub_folder_inf,
                    fields='id',
                    supportsAllDrives=True
                ).execute()
                return folder['id']
            except Exception as e:
                print(f"❌ Lỗi khi tạo thư mục {sub_folder_name}: {e}")
                return None
        return exist_folder_id

    def check_if_exists(self, drive_service, dest_folder_id, name):
        try:
            name_escaped = name.replace("'", "\\'")
            query = f"'{dest_folder_id}' in parents and name = '{name_escaped}' and trashed = false"
            results = drive_service.files().list(
                q=query,
                fields='files(id)',
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()

            if results.get('files'):
                return results['files'][0]['id']
        except Exception as e:
            print(f"❌ Lỗi khi kiểm tra tồn tại của tệp/thư mục {name}: {e}")
        return None

    def copy_multiple_files(self, drive_service, dest_folder_id, source_files):
        for source_file in source_files:
            self.copy_file(drive_service, dest_folder_id, source_file)
            time.sleep(1)  # Dừng 1 giây giữa các tệp để giảm tải API

    def extract_folder_id_from_url(self, url):
        pattern = r"[-\w]{25,}"
        match = re.search(pattern, url)
        if match:
            return match.group(0)
        else:
            print(f"❌ Không tìm thấy ID thư mục trong URL: {url}")
            return None

    def on_total_size_exceeded(self, message):
        print(message)
        sys.exit()

    def copy_drive_to_drive(self, destDriveLink, sourceDriveLink, from_page, to_page):
        service = self.get_user_credential()

        dest_folder_id = self.extract_folder_id_from_url(destDriveLink)
        source_folder_id = self.extract_folder_id_from_url(sourceDriveLink)
        if not dest_folder_id or not source_folder_id:
            print("❌ Không thể lấy ID thư mục đích hoặc nguồn.")
            return

        source_folder = service.files().get(fileId=source_folder_id, supportsAllDrives=True).execute()
        new_dest_folder_id = self.create_folder(service, dest_folder_id, source_folder['name'])

        source_files = self.get_childs_from_folder(service, source_folder_id, from_page, to_page)
        self.copy_multiple_files(service, new_dest_folder_id, source_files)

        size_gb = self._total_size / 1024
        print(f"🎉 Hoàn thành! Tổng dung lượng đã sao chép: {size_gb:.2f} GB.")

# Thiết lập và chạy
destDriveLink = dest_text.value
sourceDriveLink = source_text.value
fromPage = int(from_page_text.value)
toPage = int(to_page_text.value)

downloader = DownloadFromDrive()
downloader._limit_size = float(max_download_size_text.value)
downloader.excluded_strings = [ext.strip() for ext in exclude_str_text.value.split(",") if ext.strip()]
downloader.copy_drive_to_drive(destDriveLink, sourceDriveLink, fromPage, toPage)

📂 Tổng số tệp: 36
📂 Tổng số tệp: 3
📂 Bắt đầu sao chép thư mục Android


⚠️ Gặp lỗi giới hạn tần suất. Đợi 1.21 giây trước khi thử lại lần 1...


⚠️ Gặp lỗi giới hạn tần suất. Đợi 2.42 giây trước khi thử lại lần 2...


⚠️ Gặp lỗi giới hạn tần suất. Đợi 4.99 giây trước khi thử lại lần 3...


⚠️ Gặp lỗi giới hạn tần suất. Đợi 8.89 giây trước khi thử lại lần 4...


...

⚠️ Gặp lỗi giới hạn tần suất. Đợi 1.98 giây trước khi thử lại lần 1...


⚠️ Gặp lỗi giới hạn tần suất. Đợi 2.90 giây trước khi thử lại lần 2...


⚠️ Gặp lỗi giới hạn tần suất. Đợi 4.44 giây trước khi thử lại lần 3...


⚠️ Gặp lỗi giới hạn tần suất. Đợi 8.34 giây trước khi thử lại lần 4...


KeyboardInterrupt: 